In [6]:

!pip -q install transformers torch scikit-learn


In [7]:

import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# Sample sentences
sentences = [
    "I love learning artificial intelligence.",
    "I enjoy studying AI.",
    "The weather is very cold today."
]

# Select CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def get_embeddings(model_name):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(device)
    model.eval()

    # GPT-2 does not have a padding token by default
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.pad_token_id

    # Convert sentences into tokens
    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate token embeddings
    with torch.no_grad():
        outputs = model(**inputs)
        token_embeddings = outputs.last_hidden_state

    # Average token embeddings to get sentence embeddings
    mask = inputs["attention_mask"].unsqueeze(-1)
    mask = mask.expand(token_embeddings.size()).float()

    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    sentence_embeddings = summed / counts

    return sentence_embeddings.cpu().numpy()

# Generate BERT embeddings
print("\nGenerating BERT embeddings...")
bert_embeddings = get_embeddings("bert-base-uncased")

print("BERT embedding shape:", bert_embeddings.shape)
print("BERT embeddings:\n", bert_embeddings)

# Calculate BERT cosine similarity
bert_similarity = cosine_similarity(bert_embeddings)

print("\nBERT Cosine Similarity:")
print(bert_similarity.round(3))

# Generate GPT-2 embeddings
print("\nGenerating GPT-2 embeddings...")
gpt2_embeddings = get_embeddings("gpt2")

print("GPT-2 embedding shape:", gpt2_embeddings.shape)
print("GPT-2 embeddings:\n", gpt2_embeddings)

# Calculate GPT-2 cosine similarity
gpt2_similarity = cosine_similarity(gpt2_embeddings)

print("\nGPT-2 Cosine Similarity:")
print(gpt2_similarity.round(3))


Device: cpu

Generating BERT embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT embedding shape: (3, 768)
BERT embeddings:
 [[ 0.3780099   0.37351948 -0.035144   ... -0.23458172  0.11634245
   0.2511919 ]
 [ 0.38508153  0.13697708 -0.17118707 ...  0.04193766  0.1261985
   0.04045771]
 [ 0.09877674 -0.27174953 -0.11547562 ... -0.1364567   0.20045936
   0.1685635 ]]

BERT Cosine Similarity:
[[1.    0.834 0.567]
 [0.834 1.    0.559]
 [0.567 0.559 1.   ]]

Generating GPT-2 embeddings...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 embedding shape: (3, 768)
GPT-2 embeddings:
 [[-0.28790894  0.01638949 -0.92255706 ... -0.0238002   0.07795886
  -0.08097228]
 [-0.09526892 -0.0212316  -0.6797955  ... -0.03281324  0.11742868
   0.05747962]
 [ 0.31408605  0.06451503 -0.7138419  ...  0.38858944 -0.13061582
  -0.01685111]]

GPT-2 Cosine Similarity:
[[1.    1.    0.998]
 [1.    1.    0.998]
 [0.998 0.998 1.   ]]
